In [ ]:
import pandas as pd
import plotly.graph_objects as go
from statsmodels.tsa.stattools import acf, pacf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 9.6 MB/s eta 0:00:00


In [ ]:
df = pd.read_csv("df_final.csv", parse_dates=["timestamp"])
houses = [col for col in df.columns if col.startswith("house_")]

In [ ]:
df

,timestamp,house_1,house_2,house_3,house_4,house_5,house_6,house_7,house_8
0,2017-11-11 00:30:00,58.17,22.40,16.62,38.96,26.560,60.30,84.49,32.90
1,2017-11-11 01:00:00,49.74,19.84,14.28,33.84,22.585,55.80,74.94,29.15
2,2017-11-11 01:30:00,44.76,17.84,12.78,32.94,20.960,49.74,66.70,24.95
3,2017-11-11 02:00:00,38.22,17.32,12.06,30.30,18.800,49.56,60.67,24.60
4,2017-11-11 02:30:00,35.40,15.28,11.40,27.98,18.165,47.64,57.23,23.20
...,...,...,...,...,...,...,...,...,...
36255,2019-12-06 08:00:00,52.38,23.88,15.81,39.94,9.475,105.78,101.84,28.50
36256,2019-12-06 08:30:00,53.04,23.20,13.92,38.32,9.150,103.44,105.40,28.50
36257,2019-12-06 09:00:00,49.41,21.64,16.83,36.58,10.375,102.60,98.40,28.50
36258,2019-12-06 09:30:00,48.72,19.68,15.72,33.70,10.550,102.84,90.98,28.50


In [ ]:
#Суточный ACF
n_lags = 96

fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=houses,
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

for i, house in enumerate(houses):
    row = i // 2 + 1
    col = i % 2 + 1

    acf_values = acf(df[house], nlags=n_lags, fft=True)
    ci = 1.96 / (len(df[house]) ** 0.5)
    lags = list(range(n_lags + 1))

    fig.add_trace(go.Bar(
        x=lags, y=acf_values,
        marker_color="#1f77b4",
        showlegend=False
    ), row=row, col=col)

    fig.add_hline(y=ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)
    fig.add_hline(y=-ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)

    fig.update_xaxes(title_text="Лаг (×30 мин)", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="ACF", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="ACF: суточный горизонт (лаги 0-96, шаг 30 мин)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    showlegend=False
)

fig.show()
fig.write_image("18_acf_daily.png", scale=2)

Чётко видны пики на лагах 48 и 96 — это суточная периодичность (каждые 24 часа). Все дома показывают одинаковый паттерн — суточный цикл очень устойчив.

In [ ]:
#Недельный горизонт
n_lags = 400

fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=houses,
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

for i, house in enumerate(houses):
    row = i // 2 + 1
    col = i % 2 + 1

    acf_values = acf(df[house], nlags=n_lags, fft=True)
    ci = 1.96 / (len(df[house]) ** 0.5)
    lags = list(range(n_lags + 1))

    fig.add_trace(go.Bar(
        x=lags, y=acf_values,
        marker_color="#ff7f0e",
        showlegend=False
    ), row=row, col=col)

    fig.add_hline(y=ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)
    fig.add_hline(y=-ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)

    fig.update_xaxes(title_text="Лаг (×30 мин)", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="ACF", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="ACF: недельный горизонт (лаги 0-400, шаг 30 мин)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    showlegend=False
)

fig.show()
fig.write_image("19_acf_weekly.png", scale=2)

На графике видны чёткие пики на лагах 48, 96, 144, 192, 240, 288, 336... — это суточная периодичность (каждые 48 лагов = 24 часа). Пики постепенно затухают — корреляция слабеет с увеличением лага.

In [ ]:
#Месячный горизонт
n_lags = 1488

fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=houses,
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

for i, house in enumerate(houses):
    row = i // 2 + 1
    col = i % 2 + 1

    acf_values = acf(df[house], nlags=n_lags, fft=True)
    ci = 1.96 / (len(df[house]) ** 0.5)

    peak_lags = list(range(0, n_lags + 1, 48))
    peak_values = [acf_values[l] for l in peak_lags]

    fig.add_trace(go.Scatter(
        x=peak_lags, y=peak_values,
        mode="lines+markers",
        line=dict(color="#2ca02c", width=1.5),
        marker=dict(size=4),
        showlegend=False
    ), row=row, col=col)

    fig.add_hline(y=ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)
    fig.add_hline(y=-ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)
    fig.update_xaxes(title_text="Лаг (×30 мин)", row=row, col=col, title_font=dict(size=9))

    fig.update_yaxes(title_text="ACF", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="ACF: месячный горизонт (пики каждые 48 лагов = 1 день)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    showlegend=False
)

fig.show()
fig.write_image("20_acf_monthly.png", scale=2)

Видно что автокорреляция медленно затухает — даже на лаге 1488 (1 месяц) значения остаются на уровне 0.7-0.8. Это говорит о сильной долгосрочной памяти ряда.

In [ ]:
#Годовой горизонт
n_lags = 17520

fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=houses,
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

for i, house in enumerate(houses):
    row = i // 2 + 1
    col = i % 2 + 1

    acf_values = acf(df[house], nlags=n_lags, fft=True)
    ci = 1.96 / (len(df[house]) ** 0.5)

    peak_lags = list(range(0, n_lags + 1, 48))
    peak_values = [acf_values[l] for l in peak_lags]

    fig.add_trace(go.Scatter(
        x=peak_lags, y=peak_values,
        mode="lines+markers",
        line=dict(color="#d62728", width=1.5),
        marker=dict(size=3),
        showlegend=False
    ), row=row, col=col)

    fig.add_hline(y=ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)
    fig.add_hline(y=-ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)

    # подписи осей для каждого subplot
    fig.update_xaxes(title_text="Лаг (×30 мин)", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="ACF", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="ACF: годовой горизонт (пики каждые 48 лагов = 1 день)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    showlegend=False
)

fig.show()
fig.write_image("21_acf_yearly.png", scale=2)

Видно паттерн - ACF сначала падает до ~0.4 (около лага 8000-9000, это примерно 6 месяцев), а потом снова растёт до ~0.5. Это классическое проявление годовой сезонности - потребление в одном сезоне похоже на потребление того же сезона в прошлом году.

In [ ]:
from statsmodels.tsa.stattools import pacf

In [ ]:
#Суточный PACF
n_lags = 96

fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=houses,
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

for i, house in enumerate(houses):
    row = i // 2 + 1
    col = i % 2 + 1

    pacf_values = pacf(df[house], nlags=n_lags, method="ywm")
    ci = 1.96 / (len(df[house]) ** 0.5)
    lags = list(range(n_lags + 1))

    fig.add_trace(go.Bar(
        x=lags, y=pacf_values,
        marker_color="#9467bd",
        showlegend=False
    ), row=row, col=col)

    fig.add_hline(y=ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)
    fig.add_hline(y=-ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)

    fig.update_xaxes(title_text="Лаг (×30 мин)", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="PACF", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="PACF: суточный горизонт (лаги 0-96, шаг 30 мин)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    showlegend=False
)

fig.show()
fig.write_image("22_pacf_daily.png", scale=2)

PACF показывает очень важный результат:

лаг 1 - огромный пик (~0.9) у всех домов - текущее значение сильно зависит от предыдущего (30 минут назад)
лаг 2 - резкий отрицательный выброс - после учёта лага 1 лаг 2 даёт отрицательный вклад
лаги 3-47 - почти нулевые
лаг 48 - небольшой пик - суточная периодичность

Это говорит что для ARIMA порядок AR = 1-2, что упрощает модель.

In [ ]:
#Недельный PACF
n_lags = 400

fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=houses,
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

for i, house in enumerate(houses):
    row = i // 2 + 1
    col = i % 2 + 1

    pacf_values = pacf(df[house], nlags=n_lags, method="ywm")
    ci = 1.96 / (len(df[house]) ** 0.5)
    lags = list(range(n_lags + 1))

    fig.add_trace(go.Bar(
        x=lags, y=pacf_values,
        marker_color="#9467bd",
        showlegend=False
    ), row=row, col=col)

    fig.add_hline(y=ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)
    fig.add_hline(y=-ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)

    fig.update_xaxes(title_text="Лаг (×30 мин)", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="PACF", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="PACF: недельный горизонт (лаги 0-400, шаг 30 мин)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    showlegend=False
)

fig.show()
fig.write_image("23_pacf_weekly.png", scale=2)

In [ ]:
#Месячный PACF
n_lags = 1488

fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=houses,
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

for i, house in enumerate(houses):
    row = i // 2 + 1
    col = i % 2 + 1

    pacf_values = pacf(df[house], nlags=n_lags, method="ywm")
    ci = 1.96 / (len(df[house]) ** 0.5)
    lags = list(range(n_lags + 1))

    fig.add_trace(go.Scatter(
        x=lags, y=pacf_values,
        mode="lines",
        line=dict(color="#9467bd", width=0.8),
        showlegend=False
    ), row=row, col=col)

    fig.add_hline(y=ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)
    fig.add_hline(y=-ci, line=dict(color="red", width=1, dash="dash"), row=row, col=col)

    fig.update_xaxes(title_text="Лаг (×30 мин)", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="PACF", row=row, col=col, title_font=dict(size=9), range=[-0.2, 0.2])

fig.update_layout(
    title="PACF: месячный горизонт (лаги 0-1488, шаг 30 мин)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    showlegend=False
)

fig.show()
fig.write_image("24_pacf_monthly.png", scale=2)

Анализ автокорреляционной функции (ACF)
ACF на суточном горизонте выявил чёткие пики на лагах 48 и 96 у всех домов - математическое подтверждение суточной периодичности.
На недельном горизонте пики наблюдаются кратно лагу 48 с постепенным затуханием амплитуды.
На месячном горизонте значения ACF остаются на уровне 0.7-0.8 даже спустя 31 день - ряд обладает сильной долгосрочной памятью.
На годовом горизонте ACF демонстрирует U-образную форму с минимумом около лага 8000–9000 (6 месяцев) и последующим ростом до 0.5 - классическое проявление годовой сезонности.
Анализ частичной автокорреляционной функции (PACF)
PACF на суточном горизонте показал доминирующий пик на лаге 1 (0.9) и значимый отрицательный выброс на лаге 2.
Лаги 3-47 практически нулевые, лаг 48 показывает небольшой значимый пик - прямое влияние суточной периодичности.
На месячном горизонте PACF остаётся значимым до лага 336 (7 дней = 1 неделя), после чего обнуляется. Это означает что прямые зависимости в ряду ограничены горизонтом одной недели - всё что находится дальше передаётся через цепочку промежуточных значений, а не напрямую.
На основании результатов месячного PACF можно заключить что на горизонте свыше 336 лагов прямые зависимости отсутствуют.
Заключение:
Результаты ACF/PACF анализа дают три ключевых вывода для построения моделей прогнозирования.
Во-первых, ряд имеет AR(1) структуру с сезонной составляющей на лаге 48 — это обосновывает применение SARIMA(1,0,0)(1,0,0)[48] как базовой статистической модели.
Во-вторых, минимальное окно наблюдения для моделей должно составлять не менее 336 лагов (7 дней) - именно на этом горизонте сосредоточены все прямые зависимости.
В-третьих, сильная долгосрочная память ряда (ACF > 0.7 на горизонте месяц) свидетельствует о потенциальном преимуществе моделей с длинным окном наблюдения - LSTM, TFT, N-BEATS - перед классическими методами.